# Simple RAG (Retrieval-Augmented Generation) System

## Overview

This project implements a foundational **Retrieval-Augmented Generation (RAG)** pipeline designed for processing and querying PDF documents. By encoding document content into a vector store, the system enables efficient semantic search and retrieval of relevant information in response to natural language queries.

## Key Components

1. **PDF Processing & Text Extraction** — Ingests PDF documents and extracts raw text content for downstream processing.
2. **Text Chunking** — Splits extracted content into manageable, semantically coherent segments to improve retrieval precision.
3. **Vector Store Creation** — Generates dense vector embeddings using [HuggingFace](https://huggingface.co/) models and indexes them with [FAISS](https://engineering.fb.com/2017/03/29/data-infrastructure/faiss-a-library-for-efficient-similarity-search/) for fast similarity search.
4. **Retriever Setup** — Configures a retrieval interface to query the vector store and surface the most relevant document chunks.
5. **System Evaluation** — Assesses retrieval quality and end-to-end RAG performance against defined benchmarks.

## Method Details

### Document Preprocessing

The pipeline begins by loading the target PDF using **PyPDFLoader**, which handles the extraction of raw text while preserving the document's page structure. Once loaded, the full text is passed through **RecursiveCharacterTextSplitter**, which divides the content into overlapping chunks of a configurable size. The overlap between chunks is intentional — it ensures that sentences or ideas spanning a chunk boundary are not lost, preserving contextual continuity across adjacent segments.

### Text Cleaning

Before the chunks are embedded, a custom cleaning function — `replace_t_with_space` — is applied to each one. PDFs are notoriously inconsistent in how they encode whitespace, and tab characters (`\t`) are a common artifact of the extraction process. This step normalizes the text so that the embedding model receives clean, well-formed input rather than fragmented or misaligned strings that could degrade retrieval quality.

### Vector Store Creation

Each cleaned text chunk is passed through **HugginFace's embedding model**, which transforms the raw text into a high-dimensional vector that captures its semantic meaning. These vectors are then indexed using **FAISS** (Facebook AI Similarity Search) — a library purpose-built for fast nearest-neighbor lookups in large vector spaces. The result is a compact, queryable store where similarity between a question and a document chunk can be computed in milliseconds, regardless of document size.

### Retriever Setup

On top of the FAISS index, a retriever is configured to return the **top 2 most semantically similar chunks** for any given query. Limiting retrieval to two chunks is a deliberate tradeoff — it keeps the context window concise and reduces noise when the retrieved content is passed downstream to a language model. This value is configurable and can be tuned based on the verbosity of the source document and the specificity of expected queries.

### Encoding Function

All of the above steps are consolidated into a single function: `encode_pdf`. This function accepts a file path and a set of chunking parameters, and returns a fully initialized retriever ready for querying. By encapsulating the entire preprocessing pipeline behind one interface, the system remains easy to integrate, test, and reuse across different documents without requiring changes to downstream components.

---

## Key Features

- **Modular Design** — The `encode_pdf` function acts as a self-contained pipeline. Loading, cleaning, embedding, and indexing are all handled internally, so the caller only needs to supply a file path and parameters.
- **Configurable Chunking** — Chunk size and overlap are exposed as parameters, making it straightforward to tune the system for documents with varying structure — from dense academic papers to sparse FAQ pages.
- **Efficient Similarity Search** — FAISS is optimized for high-dimensional vector spaces and scales well beyond what a brute-force cosine similarity search could handle, making it a practical choice even as document collections grow.
- **State-of-the-Art Embeddings** — OpenAI's embedding models are trained on large, diverse corpora, giving them strong generalization across domains. This means the system can meaningfully retrieve relevant content even when the query doesn't share exact phrasing with the source text.

---

## Usage Example

To illustrate retrieval in practice, the system is tested with the query:

> *"What is the main cause of climate change?"*

The retriever searches the vector store for the two chunks most semantically aligned with this question and returns them as context. This context can then be passed directly to a language model to generate a grounded, document-specific answer — which is the core value proposition of the RAG pattern.

---

## Evaluation

The system includes an `evaluate_rag` function to benchmark retrieval performance. While the specific metrics — such as precision, recall, or mean reciprocal rank — depend on the evaluation dataset provided, the function is designed to give a structured read on how well the retriever surfaces relevant content. This is especially important when tuning chunk size or the number of retrieved results, as small parameter changes can have a meaningful impact on downstream answer quality.

---

## Benefits of this Approach

- **Scalability** — Because the document is processed in chunks rather than as a single unit, the system can handle large files without hitting memory constraints or token limits. The FAISS index grows linearly with the number of chunks, not exponentially.
- **Flexibility** — Chunk size, overlap, and the number of retrieved results are all adjustable. This makes the system adaptable to a wide range of document types and query patterns without requiring structural changes to the code.
- **Efficiency** — FAISS performs approximate nearest-neighbor search using optimized indexing structures, delivering low-latency lookups even across hundreds of thousands of vectors — far faster than exhaustive search methods.
- **Semantic Understanding** — Unlike keyword-based retrieval, embedding-based search captures *meaning* rather than surface-level word matches. A query about "carbon emissions" will correctly surface passages about "greenhouse gases" or "fossil fuel combustion," even without lexical overlap.

---

## Conclusion

This RAG system is intentionally kept simple, but it reflects sound engineering principles: clean separation of concerns, configurable parameters, and a retrieval mechanism that scales. The combination of OpenAI embeddings and FAISS creates a strong semantic search layer that can be dropped in front of virtually any language model, transforming a static PDF into an interactive, queryable knowledge source.

For teams building document-grounded question-answering tools — whether for internal knowledge bases, legal document review, or technical support — this pipeline offers a practical and extensible starting point.

# Package Installation and Imports

The cell below installs all necessary packages required to run this notebook.